# Fleet Reliability Data Engineering Platform
# Objective:
Generate synthetic datasets for vehicle fleet reliability analysis.

This notebook generates synthetic vehicle fleet reliability datasets for the Fleet Reliability Data Engineering Platform project.

## Datasets generated
- Vehicles
- Service Records
- Telemetry Summary
- Warranty Claims
- Parts Failure

## Goal
Create realistic structured datasets that can later be used for:
- ETL pipeline development
- SQL transformation
- KPI analysis
- anomaly detection
- Streamlit dashboarding

In [4]:
# Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

In [5]:
# Setting Random Seed
np.random.seed(42)
random.seed(42)

In [6]:
# Creating Output Folders
os.makedirs("data/synthetic", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [7]:
# Defining Basic Parameters
NUM_VEHICLES = 50000
NUM_SERVICE_RECORDS = 300000
NUM_TELEMETRY = 500000
NUM_WARRANTY_CLAIMS = 100000
NUM_PART_FAILURES = 120000

In [8]:
# Defining Lookup Values

models = ["Model S", "Model 3", "Model X", "Model Y", "Cybertruck"]
regions = ["North America", "Europe", "Asia Pacific", "Middle East"]
battery_types = ["LFP", "NCA", "NCM"]
fleet_types = ["Retail", "Commercial", "Internal Test"]
service_centers = ["Fremont", "Austin", "Berlin", "Shanghai", "New York", "Toronto"]
issue_categories = ["Battery", "Brakes", "Motor", "Suspension", "Software", "HVAC", "Electrical"]
severity_levels = ["Low", "Medium", "High", "Critical"]
claim_statuses = ["Approved", "Pending", "Rejected"]
failure_types = ["Wear", "Electrical Fault", "Thermal Issue", "Software Error", "Mechanical Failure"]

issue_subcategory_map = {
    "Battery": ["Battery Degradation", "Charging Failure", "Range Drop"],
    "Brakes": ["Brake Pad Wear", "Brake Noise", "ABS Fault"],
    "Motor": ["Motor Overheating", "Power Loss", "Vibration"],
    "Suspension": ["Shock Wear", "Alignment Issue", "Control Arm Fault"],
    "Software": ["UI Freeze", "Sensor Calibration", "Autopilot Error"],
    "HVAC": ["Cooling Failure", "Heating Failure", "Fan Malfunction"],
    "Electrical": ["12V Battery Issue", "Wiring Fault", "Sensor Failure"]
}

component_map = {
    "Battery": ["Battery Pack", "Charging Port", "BMS"],
    "Brakes": ["Brake Pads", "Brake Rotor", "ABS Module"],
    "Motor": ["Drive Unit", "Inverter", "Motor Controller"],
    "Suspension": ["Shock Absorber", "Control Arm", "Suspension Link"],
    "Software": ["Infotainment Module", "Autopilot Computer", "Firmware"],
    "HVAC": ["AC Compressor", "Cabin Heater", "Cooling Fan"],
    "Electrical": ["12V Battery", "Wiring Harness", "Sensor Module"]
}

In [9]:
# Helper Functions

def random_date(start_date, end_date):
    delta = end_date - start_date
    random_days = np.random.randint(0, delta.days + 1)
    return start_date + timedelta(days=int(random_days))

def generate_vin(index):
    return "5YJ" + str(index).zfill(14)

def choose_model_year():
    return np.random.choice([2019, 2020, 2021, 2022, 2023, 2024, 2025],
                            p=[0.08, 0.10, 0.14, 0.18, 0.20, 0.18, 0.12])

def get_vehicle_age_in_years(model_year):
    return max(1, 2026 - model_year)

def get_battery_health(model_year):
    base = {
        2019: 82,
        2020: 85,
        2021: 88,
        2022: 91,
        2023: 94,
        2024: 96,
        2025: 98
    }
    value = np.random.normal(base[model_year], 3)
    return round(min(100, max(70, value)), 2)

In [10]:
# Generate vehicles table

vehicle_rows = []

start_delivery = datetime(2019, 1, 1)
end_delivery = datetime(2025, 12, 31)

for i in range(1, NUM_VEHICLES + 1):
    model = np.random.choice(models, p=[0.12, 0.34, 0.10, 0.34, 0.10])
    model_year = choose_model_year()
    region = np.random.choice(regions, p=[0.45, 0.22, 0.25, 0.08])
    battery_type = np.random.choice(battery_types, p=[0.35, 0.35, 0.30])
    fleet_type = np.random.choice(fleet_types, p=[0.78, 0.17, 0.05])
    delivery_date = random_date(start_delivery, end_delivery)

    base_mileage = {
        2019: np.random.randint(70000, 180000),
        2020: np.random.randint(60000, 150000),
        2021: np.random.randint(45000, 120000),
        2022: np.random.randint(30000, 90000),
        2023: np.random.randint(15000, 60000),
        2024: np.random.randint(5000, 35000),
        2025: np.random.randint(500, 15000)
    }

    mileage = base_mileage[model_year]

    vehicle_rows.append([
        i,
        generate_vin(i),
        model,
        model_year,
        battery_type,
        delivery_date.date(),
        region,
        mileage,
        fleet_type
    ])

vehicles = pd.DataFrame(vehicle_rows, columns=[
    "vehicle_id",
    "vin",
    "model",
    "model_year",
    "battery_type",
    "delivery_date",
    "region",
    "mileage",
    "fleet_type"
])

vehicles.head()

,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type
0,1,5YJ00000000000001,Model 3,2025,NCA,2020-04-11,Asia Pacific,1269,Retail
1,2,5YJ00000000000002,Model Y,2025,NCM,2019-01-22,North America,3505,Retail
2,3,5YJ00000000000003,Model 3,2023,LFP,2024-08-30,Europe,58323,Retail
3,4,5YJ00000000000004,Model 3,2019,LFP,2025-12-31,North America,87159,Retail
4,5,5YJ00000000000005,Model 3,2023,LFP,2023-04-29,Europe,44127,Internal Test


In [11]:
# Check Vehicles

print(vehicles.shape)
vehicles.sample(5)

(50000, 9)


,vehicle_id,vin,model,model_year,battery_type,delivery_date,region,mileage,fleet_type
35916,35917,5YJ00000000035917,Model S,2021,LFP,2022-08-22,Europe,113908,Retail
31205,31206,5YJ00000000031206,Model S,2022,NCM,2025-03-20,North America,59877,Retail
10914,10915,5YJ00000000010915,Model Y,2022,NCM,2022-12-28,Europe,42401,Commercial
44564,44565,5YJ00000000044565,Model Y,2023,NCM,2025-01-25,North America,33038,Retail
37464,37465,5YJ00000000037465,Model 3,2025,LFP,2024-05-28,Asia Pacific,569,Retail


In [12]:
# Create quick lookup from vehicles

vehicle_lookup = vehicles.set_index("vehicle_id").to_dict("index")

In [13]:
# Generate service_records

service_rows = []

service_start = datetime(2021, 1, 1)
service_end = datetime(2026, 3, 1)

vehicle_ids = vehicles["vehicle_id"].values

for service_id in range(1, NUM_SERVICE_RECORDS + 1):
    vehicle_id = int(np.random.choice(vehicle_ids))
    vehicle_info = vehicle_lookup[vehicle_id]

    model_year = vehicle_info["model_year"]
    mileage = vehicle_info["mileage"]
    region = vehicle_info["region"]

    age_factor = get_vehicle_age_in_years(model_year)

    issue_category = np.random.choice(issue_categories, p=[0.18, 0.16, 0.14, 0.12, 0.18, 0.10, 0.12])
    issue_subcategory = np.random.choice(issue_subcategory_map[issue_category])
    severity = np.random.choice(severity_levels, p=[0.40, 0.32, 0.20, 0.08])

    service_date = random_date(service_start, service_end)
    service_center = np.random.choice(service_centers)

    base_cost = {
        "Battery": np.random.uniform(500, 6000),
        "Brakes": np.random.uniform(150, 1200),
        "Motor": np.random.uniform(800, 5000),
        "Suspension": np.random.uniform(250, 2200),
        "Software": np.random.uniform(50, 800),
        "HVAC": np.random.uniform(200, 1800),
        "Electrical": np.random.uniform(150, 2500)
    }

    repair_cost = base_cost[issue_category] * (1 + (age_factor - 1) * 0.07)
    downtime_days = max(0, round(np.random.normal(2.5 + age_factor * 0.3, 1.5), 1))

    repeat_issue_prob = 0.08 + (0.02 * (age_factor - 1))
    if issue_category in ["Battery", "Electrical", "Software"]:
        repeat_issue_prob += 0.04

    repeat_issue_flag = np.random.choice([0, 1], p=[1 - min(repeat_issue_prob, 0.35), min(repeat_issue_prob, 0.35)])

    service_rows.append([
        service_id,
        vehicle_id,
        service_date.date(),
        service_center,
        issue_category,
        issue_subcategory,
        severity,
        round(repair_cost, 2),
        downtime_days,
        repeat_issue_flag
    ])

service_records = pd.DataFrame(service_rows, columns=[
    "service_id",
    "vehicle_id",
    "service_date",
    "service_center",
    "issue_category",
    "issue_subcategory",
    "severity",
    "repair_cost",
    "downtime_days",
    "repeat_issue_flag"
])

service_records.head()

,service_id,vehicle_id,service_date,service_center,issue_category,issue_subcategory,severity,repair_cost,downtime_days,repeat_issue_flag
0,1,34451,2024-12-17,Fremont,Battery,Range Drop,Medium,3058.68,2.6,0
1,2,14761,2025-04-02,New York,Brakes,ABS Fault,Medium,1119.33,4.9,1
2,3,29920,2023-03-14,Berlin,Suspension,Control Arm Fault,Low,2370.96,4.2,0
3,4,46989,2024-11-06,Austin,Software,Autopilot Error,High,745.19,3.6,1
4,5,6989,2025-06-16,Fremont,Electrical,Sensor Failure,Low,1151.60,4.5,1


In [14]:
# Check service_records

print(service_records.shape)
service_records.sample(5)

(300000, 10)


,service_id,vehicle_id,service_date,service_center,issue_category,issue_subcategory,severity,repair_cost,downtime_days,repeat_issue_flag
271230,271231,35426,2023-05-23,Austin,Electrical,Sensor Failure,High,2612.31,5.8,1
170534,170535,12668,2022-12-14,Berlin,Battery,Charging Failure,Medium,689.90,4.7,0
215417,215418,6045,2021-08-26,New York,HVAC,Cooling Failure,Medium,1343.91,2.9,1
214062,214063,29628,2023-10-07,Toronto,Battery,Charging Failure,High,1850.54,3.4,0
249841,249842,1679,2022-04-14,New York,Motor,Vibration,Low,1910.92,2.1,0


In [15]:
# Generate telemetry_summary

telemetry_rows = []

telemetry_start = datetime(2025, 1, 1)
telemetry_end = datetime(2026, 3, 1)

for telemetry_id in range(1, NUM_TELEMETRY + 1):
    vehicle_id = int(np.random.choice(vehicle_ids))
    vehicle_info = vehicle_lookup[vehicle_id]

    model_year = vehicle_info["model_year"]
    model = vehicle_info["model"]
    region = vehicle_info["region"]

    event_date = random_date(telemetry_start, telemetry_end)

    battery_health = get_battery_health(model_year)

    avg_motor_temp = round(np.random.normal(72, 10), 2)
    avg_brake_temp = round(np.random.normal(58, 9), 2)

    tire_pressure_alerts = np.random.poisson(0.6)
    warning_count = np.random.poisson(1.8)

    if model_year <= 2021:
        warning_count += np.random.poisson(1.2)

    if region == "Europe":
        avg_brake_temp += np.random.normal(2, 1)

    software_version = np.random.choice([
        "2024.38.1", "2024.44.3", "2025.2.6", "2025.8.1", "2025.12.4"
    ])

    telemetry_rows.append([
        telemetry_id,
        vehicle_id,
        event_date.date(),
        battery_health,
        avg_motor_temp,
        avg_brake_temp,
        tire_pressure_alerts,
        software_version,
        int(warning_count)
    ])

telemetry_summary = pd.DataFrame(telemetry_rows, columns=[
    "telemetry_id",
    "vehicle_id",
    "date",
    "battery_health",
    "avg_motor_temp",
    "avg_brake_temp",
    "tire_pressure_alerts",
    "software_version",
    "warning_count"
])

telemetry_summary.head()

,telemetry_id,vehicle_id,date,battery_health,avg_motor_temp,avg_brake_temp,tire_pressure_alerts,software_version,warning_count
0,1,23741,2025-06-27,97.26,77.22,53.09,0,2025.8.1,7
1,2,17245,2025-03-18,98.23,72.30,62.43,2,2025.2.6,1
2,3,34950,2025-04-09,91.76,90.70,56.11,0,2025.2.6,1
3,4,49846,2025-05-31,95.87,81.47,61.18,0,2025.2.6,3
4,5,2253,2025-07-25,82.61,73.74,57.97,0,2025.12.4,2


In [16]:
# Check telemetry_summary

print(telemetry_summary.shape)
telemetry_summary.sample(5)

(500000, 9)


,telemetry_id,vehicle_id,date,battery_health,avg_motor_temp,avg_brake_temp,tire_pressure_alerts,software_version,warning_count
285792,285793,27591,2025-09-04,97.17,70.51,51.42,0,2024.44.3,1
382845,382846,22505,2025-05-31,96.04,91.79,46.51,1,2025.2.6,5
257544,257545,30382,2025-03-31,90.72,68.40,48.89,0,2024.44.3,5
83441,83442,10407,2025-02-01,94.17,79.92,67.64,1,2024.38.1,2
113900,113901,38712,2025-08-21,94.29,58.36,47.69,1,2025.2.6,3


In [17]:
# Generate warranty_claims

warranty_rows = []

claim_start = datetime(2021, 1, 1)
claim_end = datetime(2026, 3, 1)

for claim_id in range(1, NUM_WARRANTY_CLAIMS + 1):
    vehicle_id = int(np.random.choice(vehicle_ids))
    vehicle_info = vehicle_lookup[vehicle_id]

    model_year = vehicle_info["model_year"]
    age_factor = get_vehicle_age_in_years(model_year)

    category = np.random.choice(issue_categories, p=[0.20, 0.12, 0.15, 0.10, 0.16, 0.10, 0.17])
    component = np.random.choice(component_map[category])

    claim_date = random_date(claim_start, claim_end)

    claim_amount = {
        "Battery": np.random.uniform(1000, 9000),
        "Brakes": np.random.uniform(100, 1800),
        "Motor": np.random.uniform(900, 6500),
        "Suspension": np.random.uniform(250, 2500),
        "Software": np.random.uniform(50, 1000),
        "HVAC": np.random.uniform(200, 2200),
        "Electrical": np.random.uniform(150, 2800)
    }[category]

    claim_amount = claim_amount * (1 + (age_factor - 1) * 0.05)
    claim_status = np.random.choice(claim_statuses, p=[0.68, 0.20, 0.12])

    warranty_rows.append([
        claim_id,
        vehicle_id,
        claim_date.date(),
        component,
        round(claim_amount, 2),
        claim_status
    ])

warranty_claims = pd.DataFrame(warranty_rows, columns=[
    "claim_id",
    "vehicle_id",
    "claim_date",
    "component",
    "claim_amount",
    "claim_status"
])

warranty_claims.head()

,claim_id,vehicle_id,claim_date,component,claim_amount,claim_status
0,1,28788,2022-08-11,Wiring Harness,2325.68,Pending
1,2,46239,2023-11-26,BMS,7554.90,Approved
2,3,11206,2022-08-05,Drive Unit,5738.37,Pending
3,4,23091,2022-01-30,Drive Unit,7381.06,Approved
4,5,521,2023-11-23,Cabin Heater,2123.84,Approved


In [18]:
# Check warranty_claims

print(warranty_claims.shape)
warranty_claims.sample(5)

(100000, 6)


,claim_id,vehicle_id,claim_date,component,claim_amount,claim_status
21110,21111,16693,2025-09-05,Battery Pack,2587.58,Approved
7226,7227,30639,2022-08-05,Infotainment Module,1077.10,Approved
59993,59994,22475,2025-01-29,Charging Port,9795.71,Approved
67235,67236,46237,2021-02-07,Drive Unit,5728.27,Rejected
21301,21302,19722,2022-05-13,Inverter,6227.83,Pending


In [19]:
# Generate parts_failure

parts_failure_rows = []

failure_start = datetime(2021, 1, 1)
failure_end = datetime(2026, 3, 1)

for failure_id in range(1, NUM_PART_FAILURES + 1):
    vehicle_id = int(np.random.choice(vehicle_ids))
    vehicle_info = vehicle_lookup[vehicle_id]

    model_year = vehicle_info["model_year"]
    age_factor = get_vehicle_age_in_years(model_year)

    category = np.random.choice(issue_categories, p=[0.20, 0.14, 0.15, 0.11, 0.16, 0.10, 0.14])
    component = np.random.choice(component_map[category])
    failure_date = random_date(failure_start, failure_end)

    if category == "Software":
        failure_type = "Software Error"
    elif category == "Battery":
        failure_type = np.random.choice(["Thermal Issue", "Electrical Fault", "Wear"])
    else:
        failure_type = np.random.choice(failure_types)

    replaced_flag = np.random.choice([0, 1], p=[0.25, 0.75])

    parts_failure_rows.append([
        failure_id,
        vehicle_id,
        component,
        failure_date.date(),
        failure_type,
        replaced_flag
    ])

parts_failure = pd.DataFrame(parts_failure_rows, columns=[
    "failure_id",
    "vehicle_id",
    "component",
    "failure_date",
    "failure_type",
    "replaced_flag"
])

parts_failure.head()

,failure_id,vehicle_id,component,failure_date,failure_type,replaced_flag
0,1,5821,Cabin Heater,2025-08-25,Thermal Issue,1
1,2,13420,ABS Module,2025-06-16,Thermal Issue,1
2,3,44,Cooling Fan,2026-02-09,Software Error,1
3,4,27146,Motor Controller,2021-03-04,Software Error,0
4,5,48454,Wiring Harness,2021-02-05,Thermal Issue,0


In [20]:
# Check parts_failure
print(parts_failure.shape)
parts_failure.sample(5)

(120000, 6)


,failure_id,vehicle_id,component,failure_date,failure_type,replaced_flag
116392,116393,42891,Infotainment Module,2025-03-02,Software Error,1
23553,23554,8078,Motor Controller,2024-05-14,Mechanical Failure,0
55614,55615,40616,Battery Pack,2023-03-04,Electrical Fault,1
58203,58204,44366,Battery Pack,2021-12-08,Electrical Fault,1
118100,118101,85,Brake Pads,2025-06-16,Software Error,0


In [22]:
# Basic Validation Checks
print("Vehicles null values:")
print(vehicles.isnull().sum())
print()

print("Service records null values:")
print(service_records.isnull().sum())
print()

print("Telemetry summary null values:")
print(telemetry_summary.isnull().sum())
print()

print("Warranty claims null values:")
print(warranty_claims.isnull().sum())
print()

print("Parts failure null values:")
print(parts_failure.isnull().sum())

Vehicles null values:
vehicle_id       0
vin              0
model            0
model_year       0
battery_type     0
delivery_date    0
region           0
mileage          0
fleet_type       0
dtype: int64

Service records null values:
service_id           0
vehicle_id           0
service_date         0
service_center       0
issue_category       0
issue_subcategory    0
severity             0
repair_cost          0
downtime_days        0
repeat_issue_flag    0
dtype: int64

Telemetry summary null values:
telemetry_id            0
vehicle_id              0
date                    0
battery_health          0
avg_motor_temp          0
avg_brake_temp          0
tire_pressure_alerts    0
software_version        0
warning_count           0
dtype: int64

Warranty claims null values:
claim_id        0
vehicle_id      0
claim_date      0
component       0
claim_amount    0
claim_status    0
dtype: int64

Parts failure null values:
failure_id       0
vehicle_id       0
component        0
failur

In [23]:
# Check Duplicates

print("Duplicate rows check")
print("vehicles:", vehicles.duplicated().sum())
print("service_records:", service_records.duplicated().sum())
print("telemetry_summary:", telemetry_summary.duplicated().sum())
print("warranty_claims:", warranty_claims.duplicated().sum())
print("parts_failure:", parts_failure.duplicated().sum())

Duplicate rows check
vehicles: 0
service_records: 0
telemetry_summary: 0
warranty_claims: 0
parts_failure: 0


In [24]:
# Quick Summary

print("Vehicles by model")
print(vehicles["model"].value_counts())
print()

print("Service issue categories")
print(service_records["issue_category"].value_counts())
print()

print("Warranty claim status")
print(warranty_claims["claim_status"].value_counts())
print()

print("Top failed components")
print(parts_failure["component"].value_counts().head(10))

Vehicles by model
model
Model Y       17100
Model 3       16909
Model S        6090
Cybertruck     4970
Model X        4931
Name: count, dtype: int64

Service issue categories
issue_category
Software      53964
Battery       53790
Brakes        47883
Motor         42029
Suspension    36315
Electrical    35937
HVAC          30082
Name: count, dtype: int64

Warranty claim status
claim_status
Approved    67973
Pending     20058
Rejected    11969
Name: count, dtype: int64

Top failed components
component
Battery Pack           8010
BMS                    8003
Charging Port          7983
Autopilot Computer     6516
Firmware               6400
Infotainment Module    6392
Drive Unit             6018
Motor Controller       5993
Inverter               5867
Brake Rotor            5672
Name: count, dtype: int64


In [25]:
# Saving as CSV

vehicles.to_csv("data/synthetic/vehicles.csv", index=False)
service_records.to_csv("data/synthetic/service_records.csv", index=False)
telemetry_summary.to_csv("data/synthetic/telemetry_summary.csv", index=False)
warranty_claims.to_csv("data/synthetic/warranty_claims.csv", index=False)
parts_failure.to_csv("data/synthetic/parts_failure.csv", index=False)

print("All synthetic datasets saved successfully.")

All synthetic datasets saved successfully.


In [27]:
# Save copy into raw folder
vehicles.to_csv("data/raw/vehicles.csv", index=False)
service_records.to_csv("data/raw/service_records.csv", index=False)
telemetry_summary.to_csv("data/raw/telemetry_summary.csv", index=False)
warranty_claims.to_csv("data/raw/warranty_claims.csv", index=False)
parts_failure.to_csv("data/raw/parts_failure.csv", index=False)

print("Raw data files saved successfully.")


Raw data files saved successfully.


In [28]:
# Final shape check

print("Final dataset shapes")
print("vehicles:", vehicles.shape)
print("service_records:", service_records.shape)
print("telemetry_summary:", telemetry_summary.shape)
print("warranty_claims:", warranty_claims.shape)
print("parts_failure:", parts_failure.shape)

Final dataset shapes
vehicles: (50000, 9)
service_records: (300000, 10)
telemetry_summary: (500000, 9)
warranty_claims: (100000, 6)
parts_failure: (120000, 6)
